In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

# Check columns
print(sample.head())
print(sample.columns)

# Dummy prediction
sample.iloc[:, 1] = "A"

# Save submission file
sample.to_csv("/kaggle/working/submission.csv", index=False)

print("submission.csv created successfully!")

   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C
Index(['ID', 'Prediction'], dtype='object')
submission.csv created successfully!


# EDA

In [28]:
import numpy as np
import pandas as pd
import re
import string
import os

SEED = 42
np.random.seed(SEED)

In [4]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [5]:
print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [6]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [7]:
print(train.info())
print(train.isnull().sum())
print(train['answer'].value_counts())
print(train.duplicated().sum())
print(train.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
0
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   

                                               

# Text Cleaning

In [30]:
from sklearn.model_selection import train_test_split

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

for col in text_cols:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

train[['prompt', 'A']].head()

Train: (1600, 9) Validation: (400, 9)


# Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

options = ['A', 'B', 'C', 'D', 'E']

tr, val = train_test_split(
    train,
    test_size=0.2,
    random_state=SEED,
    stratify=train['answer']
)

print("Train:", tr.shape, "Validation:", val.shape)

# mAP@3 Scoring

In [11]:
def apk(actual, predicted, k=3):
    """Average precision for a single example with one correct answer."""
    predicted = predicted[:k]
    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0

In [22]:
def mapk(actuals, predictions, k=3):
    """Mean average precision @ k. `predictions` is a list/Series of
    space-separated letter strings, e.g. 'A B C'."""
    scores = [
        apk(actual, pred.split(), k)
        for actual, pred in zip(actuals, predictions)
    ]
    return np.mean(scores)

In [23]:
assert apk('A', ['A', 'B', 'C']) == 1.0
assert apk('B', ['A', 'B', 'C']) == 0.5
assert apk('D', ['A', 'B', 'C']) == 0.0
print("mAP@3 helper functions OK")

mAP@3 helper functions OK


## Random baseline

Worth establishing *before* looking at any model: with 5 options and a top-3
guess, a uniformly random ranking already scores fairly high on mAP@3 purely by
chance. Any real method needs to clear this bar to be worth using.

In [31]:
rng = np.random.default_rng(SEED)

random_preds = [
    " ".join(rng.permutation(options)[:3])
    for _ in range(len(val))
]

random_score = mapk(val['answer'], random_preds)
print("Random baseline mAP@3 (validation):", round(random_score, 4))

Random baseline mAP@3 (validation): 0.3208


# TF-IDF + Cosine Similarity

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [51]:
all_text = pd.concat([train[c] for c in text_cols] + [test[c] for c in text_cols])

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf.fit(all_text)

print("Vocabulary size:", len(tfidf.vocabulary_))

Vocabulary size: 2865


In [33]:
def predict_top3_tfidf(df, vectorizer):
  
    prompt_mat = vectorizer.transform(df['prompt'])
    option_mats = {opt: vectorizer.transform(df[opt]) for opt in options}

 
    sims = np.column_stack([
        cosine_similarity(prompt_mat, option_mats[opt]).diagonal()
        for opt in options
    ])  

    rankings = np.argsort(-sims, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    return preds

In [34]:
val_preds_tfidf = predict_top3_tfidf(val, tfidf)
tfidf_score = mapk(val['answer'], val_preds_tfidf)
print("TF-IDF mAP@3 (validation):", round(tfidf_score, 4))
print("Random baseline   mAP@3 (validation):", round(random_score, 4))

TF-IDF mAP@3 (validation): 0.2821
Random baseline   mAP@3 (validation): 0.3208


# Word2Vec Embeddings

In [12]:
from gensim.models import Word2Vec

import gensim
print(gensim.__version__)

4.4.0


## Tokenize Text

In [13]:
def tokenize(text):
    return str(text).split()

In [25]:
sentences = []
for col in text_cols:
    sentences.extend(train[col].apply(tokenize).tolist())
    sentences.extend(test[col].apply(tokenize).tolist())

print("Number of sentences:", len(sentences))

Number of sentences: 15000


## Training Word2Vec

In [35]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=SEED
)

print("Vocab size:", len(w2v_model.wv))

Vocab size: 3096


In [16]:
print(w2v_model.wv.vector_size)

100


## Sentence Embeddings

In [36]:
def sentence_vector(text, model):
    words = str(text).split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if not vectors:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

## Cosine Similarity using Word2Vec

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

options = ['A','B','C','D','E']

def predict_top3_w2v(df, model):
    prompt_vecs = np.array([sentence_vector(p, model) for p in df['prompt']])
    option_vecs = {
        opt: np.array([sentence_vector(t, model) for t in df[opt]])
        for opt in options
    }

    sims = np.column_stack([
        cosine_similarity(prompt_vecs, option_vecs[opt]).diagonal()
        for opt in options
    ])

    rankings = np.argsort(-sims, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    return preds

## Evaluate

In [38]:
val_preds_w2v = predict_top3_w2v(val, w2v_model)
w2v_score = mapk(val['answer'], val_preds_w2v)
print("Word2Vec mAP@3 (validation):", round(w2v_score, 4))

Word2Vec mAP@3 (validation): 0.3367


# BERT, RoBERTa and Attention Mechanism

## BERT

BERT (Bidirectional Encoder Representations from Transformers) is a pretrained transformer model that learns contextual representations by processing text bidirectionally. Unlike traditional word embeddings, BERT considers both left and right context when representing each word.

## RoBERTa

RoBERTa (Robustly Optimized BERT Approach) is an improved version of BERT. It is trained on more data, for more iterations, and removes the Next Sentence Prediction objective, resulting in better performance on many NLP tasks.

## Self-Attention

The transformer architecture uses self-attention to determine how much each word should focus on every other word in a sentence. This allows the model to capture long-range dependencies and understand context effectively.

## Context-Aware Embeddings

Unlike Word2Vec, which assigns a fixed vector to every word, transformer models generate contextual embeddings. The same word can have different vector representations depending on its surrounding words, leading to better semantic understanding.

# Transformers

In [39]:
!pip install -q sentence-transformers

In [40]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [41]:
def predict_top3_transformer(df, model, batch_size=64):
    prompt_emb = model.encode(
        df['prompt'].tolist(),
        convert_to_numpy=True,
        batch_size=batch_size,
        show_progress_bar=False,
    )
    option_embs = {
        opt: model.encode(
            df[opt].tolist(),
            convert_to_numpy=True,
            batch_size=batch_size,
            show_progress_bar=False,
        )
        for opt in options
    }

    sims = np.column_stack([
        cosine_similarity(prompt_emb, option_embs[opt]).diagonal()
        for opt in options
    ])

    rankings = np.argsort(-sims, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    return preds

In [42]:
val_preds_transformer = predict_top3_transformer(val, model)
transformer_score = mapk(val['answer'], val_preds_transformer)
print("MiniLM mAP@3 (validation):", round(transformer_score, 4))

MiniLM mAP@3 (validation): 0.3925


## Comparing the embedding-based models so far

In [43]:
results = pd.DataFrame({
    "Model": ["Random baseline", "TF-IDF", "Word2Vec", "MiniLM"],
    "mAP@3 (validation)": [random_score, tfidf_score, w2v_score, transformer_score],
}).sort_values("mAP@3 (validation)", ascending=False).reset_index(drop=True)

results

,Model,mAP@3 (validation)
0,MiniLM,0.392500
1,Word2Vec,0.336667
2,Random baseline,0.320833
3,TF-IDF,0.282083


# Zero-Shot Classification

In [44]:
from transformers import pipeline

In [45]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [46]:
def predict_zero_shot(row, classifier):
    candidate_labels = [row[o] for o in options]

    result = classifier(row["prompt"], candidate_labels)

    
    label_to_letter = {i: opt for i, opt in enumerate(options)}
    ranking = [
        label_to_letter[candidate_labels.index(label)]
        for label in result["labels"]
    ]
    return " ".join(ranking[:3])

In [47]:
zero_shot_sample = val.head(20).copy()

zero_shot_sample["zero_prediction"] = zero_shot_sample.apply(
    lambda row: predict_zero_shot(row, zero_shot),
    axis=1
)

zero_score = mapk(zero_shot_sample["answer"], zero_shot_sample["zero_prediction"])
print("Zero-Shot mAP@3 (20-row sample):", round(zero_score, 4))

Zero-Shot mAP@3 (20-row sample): 0.525


In [48]:
comparison = pd.DataFrame({
    "Model": [
        "Random baseline",
        "TF-IDF",
        "Word2Vec",
        "MiniLM",
        "Zero-Shot (20-row sample only — not directly comparable)",
    ],
    "mAP@3": [random_score, tfidf_score, w2v_score, transformer_score, zero_score],
})

comparison

,Model,mAP@3
0,Random baseline,0.320833
1,TF-IDF,0.282083
2,Word2Vec,0.336667
3,MiniLM,0.392500
4,Zero-Shot (20-row sample only — not directly c...,0.525000


# sub

In [49]:
best_model_name = results.iloc[0]["Model"]
print("Best model on validation:", best_model_name)


test_predictions = predict_top3_transformer(test, model)

Best model on validation: MiniLM


In [50]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions,
})

submission.to_csv("submission.csv", index=False)
submission.head()

,ID,Prediction
0,1,B E A
1,2,B E D
2,3,A C D
3,4,E A C
4,5,B D C
